<a href="https://colab.research.google.com/github/anamacao/FAPESP-PIBIC-scrapping/blob/main/UE/charts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GRÁFICOS

Este Colab usa a base semanal acumulada em `data/news.csv` no GitHub. Se `internet_governance_news.db` estiver disponível na sessão, os gráficos gerais continuam usando esse banco; os dois gráficos semanais usam o CSV. A coleta automática cobre inicialmente NIC.br, EDPB, Mercociudades e Senado Federal. Confira as falhas no relatório antes de interpretar uma queda.


In [ ]:
# %%
import sqlite3
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.io as pio
import requests
import re

pio.renderers.default = "notebook_connected"

DATABASE_NAME = "internet_governance_news.db"
NEWS_URL = "https://raw.githubusercontent.com/anamacao/FAPESP-PIBIC-scrapping/main/data/news.csv"
REPORT_URL = "https://raw.githubusercontent.com/anamacao/FAPESP-PIBIC-scrapping/main/data/runs/latest.json"
SOURCE_NAMES = {
    "nic": "NIC.br", "edpb": "EDPB",
    "mercociudades": "Mercociudades", "senado_federal": "Senado Federal",
}


def load_news():
    news = pd.read_csv(NEWS_URL, dtype=str, keep_default_na=False)
    required = {"source_id", "title", "published_date", "url", "first_seen"}
    if not required.issubset(news.columns):
        raise ValueError(f"Colunas ausentes no CSV: {sorted(required - set(news.columns))}")
    news["source"] = news["source_id"].map(SOURCE_NAMES).fillna(news["source_id"])
    return news.drop_duplicates(subset=["source_id", "url"])


df_news = load_news()


def load_articles():
    if Path(DATABASE_NAME).is_file():
        with sqlite3.connect(DATABASE_NAME) as conn:
            exists = conn.execute(
                "SELECT 1 FROM sqlite_master WHERE type = 'table' AND name = 'articles'"
            ).fetchone()
            if exists:
                articles = pd.read_sql("SELECT * FROM articles", conn)
                if not articles.empty:
                    print("📦 Gráficos gerais: banco SQLite desta sessão.")
                    return articles
    print("📦 Gráficos gerais: CSV acumulado do GitHub (4 fontes iniciais).")
    return df_news.rename(columns={"published_date": "date"}).copy()


df_all = load_articles()
print(f"📰 {len(df_news)} notícias no CSV; {len(df_all)} registros para os gráficos gerais.")
df_all.head()


### Preparo das datas

In [ ]:
# %%
df_time = df_all.copy()
raw_dates = df_time["date"].fillna("").astype(str)
iso = raw_dates.str.extract(r"(\d{4}-\d{2}-\d{2})")[0]
br = raw_dates.str.extract(r"(\d{1,2}/\d{1,2}/\d{4})")[0]
df_time["date_clean"] = pd.to_datetime(iso, format="%Y-%m-%d", errors="coerce")
df_time["date_clean"] = df_time["date_clean"].fillna(
    pd.to_datetime(br, format="%d/%m/%Y", errors="coerce")
)
df_time = df_time.dropna(subset=["date_clean"])
print(f"⏱️ Registros com data válida: {len(df_time)}")


### Contagem de notícias por fonte (base carregada)


In [ ]:
df_all["source"].value_counts()

### Distribuição de notícias por fonte (base carregada)


In [ ]:

source_counts = df_all["source"].value_counts().reset_index()
source_counts.columns = ["source", "count"]

px.pie(
    source_counts,
    names="source",
    values="count",
    title="Distribuição de Notícias por Fonte (Todos os Sites)"
).show()


### Total de notícias ao longo do tempo

In [ ]:

timeline = (
    df_time["date_clean"]
    .value_counts()
    .sort_index()
    .reset_index()
)

timeline.columns = ["data", "quantidade"]

px.line(
    timeline,
    x="data",
    y="quantidade",
    title="Total de Notícias ao Longo do Tempo"
).show()


### Top 15 notícias mais recentes

In [ ]:
# %%
top15 = (
    df_time.sort_values("date_clean", ascending=False)
           .head(15)
           .copy()
)

top15["ordem"] = range(1, len(top15) + 1)

px.bar(
    top15,
    x="ordem",
    y="title",
    orientation="h",
    title="Top 15 Notícias Mais Recentes"
).show()


### Palavras mais frequentes nos títulos (base carregada)


In [ ]:
# %%
text = " ".join(df_all["title"].astype(str)).lower()
words = re.findall(r"\b\w{4,}\b", text)

word_freq = (
    pd.Series(words)
    .value_counts()
    .head(20)
    .reset_index()
)

word_freq.columns = ["palavra", "frequencia"]

px.treemap(
    word_freq,
    path=["palavra"],
    values="frequencia",
    title="Palavras Mais Frequentes nos Títulos (Todos os Sites)"
).show()


### Comparação direta entre fontes específicas

In [ ]:
# %%
compare = (
    df_all["source"]
    .value_counts()
    .reset_index()
)

compare.columns = ["fonte", "quantidade"]

px.bar(
    compare,
    x="fonte",
    y="quantidade",
    title="Comparação de Volume de Notícias por Fonte"
).show()


## Notícias publicadas nos últimos sete dias

Os gráficos abaixo usam a **data de publicação**, não o dia em que o link entrou no CSV. Os registros anteriores continuam na base histórica. Uma fonte com falha pode deixar a semana incompleta.


In [ ]:
# %%
hoje = pd.Timestamp.now(tz="America/Sao_Paulo").normalize().tz_localize(None)
inicio = hoje - pd.Timedelta(days=6)
df_week = df_news.copy()
df_week["date_clean"] = pd.to_datetime(
    df_week["published_date"], format="%Y-%m-%d", errors="coerce"
)
df_week = df_week[df_week["date_clean"].between(inicio, hoje)].copy()
print(f"📅 {inicio:%d/%m/%Y} a {hoje:%d/%m/%Y}: {len(df_week)} notícias publicadas")
print(f"Sem data reconhecida no CSV: {pd.to_datetime(df_news['published_date'], format='%Y-%m-%d', errors='coerce').isna().sum()}")
try:
    response = requests.get(REPORT_URL, timeout=20)
    response.raise_for_status()
    report = response.json()
    failed = [f"{s['source_id']}: {s.get('error', 'falha')}" for s in report["sources"] if s["status"] != "ok"]
    print(f"Última coleta: {report['generated_at']}")
    if failed:
        print("⚠️ Cobertura parcial — " + "; ".join(failed))
except (requests.RequestException, ValueError, KeyError) as exc:
    print(f"⚠️ Não foi possível consultar o diagnóstico da coleta: {exc}")


### Publicações da semana por fonte


In [ ]:
# %%
if df_week.empty:
    print("Não há notícias datadas nesta janela; verifique o relatório da coleta.")
else:
    weekly_sources = (df_week.groupby("source").size()
                      .reindex(SOURCE_NAMES.values(), fill_value=0)
                      .rename_axis("source").reset_index(name="noticias")
                      .sort_values("noticias", ascending=False))
    fig = px.bar(weekly_sources, x="source", y="noticias", text="noticias",
                 title=f"Notícias publicadas por fonte — {inicio:%d/%m} a {hoje:%d/%m/%Y}",
                 labels={"source": "Fonte", "noticias": "Notícias"})
    fig.update_traces(textposition="outside")
    fig.show()


### Publicações por dia na semana


In [ ]:
# %%
if df_week.empty:
    print("Sem dados suficientes para o gráfico diário desta semana.")
else:
    daily = (df_week.groupby("date_clean").size()
             .reindex(pd.date_range(inicio, hoje), fill_value=0)
             .rename_axis("data").reset_index(name="noticias"))
    fig = px.line(daily, x="data", y="noticias", markers=True,
                  title=f"Notícias por dia — {inicio:%d/%m} a {hoje:%d/%m/%Y}",
                  labels={"data": "Data de publicação", "noticias": "Notícias"})
    fig.show()
